# DQ-03 · order_items.csv
Checks: null rate, duplicates, FK → orders / products / promotions, business rules (price formulas).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
items  = pd.read_csv('order_items.csv', low_memory=False)
orders = pd.read_csv('orders.csv')
prods  = pd.read_csv('products.csv')
promos = pd.read_csv('promotions.csv')
print(f'Shape: {items.shape}')
items.head(3)

## 1. Null rate

In [ ]:
null_counts = items.isnull().sum()
print(pd.DataFrame({'null_count': null_counts, 'null_%': (null_counts/len(items)*100).round(2)}))

## 2. Duplicate (order_id, product_id)

In [ ]:
# Một order có thể có cùng product_id nhiều lần (hợp lệ nếu dòng khác nhau về qty/promo)
dup = items.duplicated(subset=['order_id','product_id'], keep=False)
flag('Duplicate (order_id, product_id) rows', dup, items)

## 3. FK: order_id → orders

In [ ]:
valid_orders = set(orders['order_id'])
flag('order_id not in orders', ~items['order_id'].isin(valid_orders), items)

## 4. FK: product_id → products

In [ ]:
valid_prods = set(prods['product_id'])
flag('product_id not in products', ~items['product_id'].isin(valid_prods), items)

## 5. FK: promo_id → promotions (where not null)

In [ ]:
valid_promos = set(promos['promo_id'])
has_promo    = items['promo_id'].notna()
flag('promo_id not in promotions', has_promo & ~items['promo_id'].isin(valid_promos), items)

## 6. promo_id_2 — entirely null?

In [ ]:
print(f'promo_id_2 non-null: {items["promo_id_2"].notna().sum()}')

## 7. Business rules: quantity, unit_price, discount_amount

In [ ]:
flag('quantity ≤ 0',         items['quantity'] <= 0,         items)
flag('unit_price ≤ 0',       items['unit_price'] <= 0,       items)
flag('discount_amount < 0',  items['discount_amount'] < 0,   items)
flag('discount_amount > quantity × unit_price',
     items['discount_amount'] > items['quantity'] * items['unit_price'], items)

## 8. Business rule: discount formula

In [ ]:
df = items.merge(promos[['promo_id','promo_type','discount_value']], on='promo_id', how='left')

# percentage: discount_amount = qty × unit_price × (disc_val/100)
pct = df[df['promo_type']=='percentage'].copy()
pct['expected'] = pct['quantity'] * pct['unit_price'] * (pct['discount_value']/100)
pct['diff']     = (pct['discount_amount'] - pct['expected']).abs()
flag('percentage promo: discount_amount error > 1.0', pct['diff'] > 1.0, pct)

# fixed: discount_amount = qty × discount_value
fix = df[df['promo_type']=='fixed'].copy()
fix['expected'] = fix['quantity'] * fix['discount_value']
fix['diff']     = (fix['discount_amount'] - fix['expected']).abs()
flag('fixed promo: discount_amount error > 0.01', fix['diff'] > 0.01, fix)

# no promo → discount_amount phải = 0
no_p = df[df['promo_id'].isna()].copy()
flag('no promo but discount_amount > 0', no_p['discount_amount'] > 0, no_p)

## Summary

In [ ]:
summary()